## COMMODITIES

In [ ]:
"""
Commodity Data Collection Script
Sources used (in priority order):
  1. yfinance  — ETF/futures proxies freely available for most commodities
  2. EODHD     — if EODHD_API_KEY env var is set
  3. OilPrice  — if OILPRICEAPI_KEY env var is set (energy only)

Every commodity has at least one yfinance ticker so the script works
with zero API keys while API-sourced data is used to supplement when
available.
"""

import os
import sys
import subprocess
import time
import logging
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def _pip_install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])


for _pkg in ['pandas', 'numpy', 'tqdm', 'yfinance', 'requests', 'python-dotenv']:
    try:
        __import__(_pkg.replace('-', '_'))
    except ImportError:
        logger.info(f"Installing {_pkg}…")
        _pip_install(_pkg)

import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
import yfinance as yf
from dotenv import load_dotenv

load_dotenv()

OHLCV = ['Open', 'High', 'Low', 'Close', 'Volume']


def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def _standardise(df: pd.DataFrame, price_col: str = 'Close') -> pd.DataFrame:
    """Ensure OHLCV columns are present and index is DatetimeIndex named 'Date'."""
    if df is None or df.empty:
        return pd.DataFrame()
    df = _flatten_columns(df.copy())
    # Case-insensitive rename
    lmap = {c.lower(): c for c in df.columns}
    rename = {}
    for want in OHLCV:
        if want not in df.columns and want.lower() in lmap:
            rename[lmap[want.lower()]] = want
    if rename:
        df.rename(columns=rename, inplace=True)

    # If only a single price column exists, create synthetic OHLC
    if 'Close' not in df.columns:
        for candidate in [price_col, 'Price', 'Value', 'price', 'value']:
            if candidate in df.columns:
                df['Close'] = df[candidate]
                break
    if 'Close' not in df.columns:
        return pd.DataFrame()

    for c in ['Open', 'High', 'Low']:
        if c not in df.columns:
            df[c] = df['Close']
    if 'Volume' not in df.columns:
        df['Volume'] = 0

    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = 'Date'
    return df[OHLCV].sort_index()


# ---------------------------------------------------------------------------
# Commodity definitions
# yf_tickers: list of yfinance symbols to try in order (first success wins)
# ---------------------------------------------------------------------------
COMMODITIES = {
    'Energy': {
        'Brent_Crude_Oil': {
            'yf_tickers': ['BZ=F', 'BNO'],           # Brent futures / Brent ETF
            'eodhd_code': 'BRENT.COMM',
        },
        'WTI_Crude_Oil': {
            'yf_tickers': ['CL=F', 'USO'],            # WTI futures / US Oil ETF
            'eodhd_code': 'WTI.COMM',
        },
        'Natural_Gas': {
            'yf_tickers': ['NG=F', 'UNG'],            # Nat gas futures / ETF
            'eodhd_code': 'NATURALGAS.COMM',
        },
    },
    'Precious_Metals': {
        'Gold': {
            'yf_tickers': ['GC=F', 'GLD', 'IAU'],    # Gold futures / ETFs
            'eodhd_code': 'XAUUSD.FOREX',
        },
        'Silver': {
            'yf_tickers': ['SI=F', 'SLV'],
            'eodhd_code': 'XAGUSD.FOREX',
        },
        'Platinum': {
            'yf_tickers': ['PL=F', 'PPLT'],
            'eodhd_code': None,
        },
    },
    'Base_Metals': {
        'Copper': {
            'yf_tickers': ['HG=F', 'CPER'],          # Copper futures / ETF
            'eodhd_code': None,
        },
        'Aluminum': {
            'yf_tickers': ['ALI=F'],                  # Aluminum futures
            'eodhd_code': None,
        },
        'Lead': {
            'yf_tickers': ['LL=F'],                   # Lead futures
            'eodhd_code': None,
        },
    },
    'Agriculture': {
        'Wheat': {
            'yf_tickers': ['ZW=F', 'WEAT'],          # Wheat futures / ETF
            'eodhd_code': None,
        },
        'Corn': {
            'yf_tickers': ['ZC=F', 'CORN'],          # Corn futures / ETF
            'eodhd_code': None,
        },
        'Cotton': {
            'yf_tickers': ['CT=F', 'BAL'],           # Cotton futures / ETF
            'eodhd_code': None,
        },
        'Natural_Rubber': {
            'yf_tickers': ['CEAT.NS', 'MRF.NS', 'APOLLOTYRE.NS'],  # Indian tyre cos = rubber price proxies
            'eodhd_code': None,
        },
    },
    'Fertilizers': {
        'Fertilizer_Index': {
            'yf_tickers': ['MOO', 'MOS', 'NTR'],     # Agri / fertiliser proxies
            'eodhd_code': None,
        },
    },
}


class CommodityDataCollector:
    def __init__(self, start_date='2021-01-01', end_date=None, output_dir='Commodities'):
        self.start_date = datetime.strptime(start_date, '%Y-%m-%d')
        self.end_date = datetime.strptime(
            end_date or datetime.today().strftime('%Y-%m-%d'), '%Y-%m-%d')
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        self.eodhd_key = os.getenv('EODHD_API_KEY')
        if not self.eodhd_key:
            logger.warning("EODHD_API_KEY not set – skipping EODHD source")

        logger.info(f"Commodity collector ready  ·  "
                    f"{self.start_date.date()} → {self.end_date.date()}")

    # ------------------------------------------------------------------
    # yfinance
    # ------------------------------------------------------------------
    def _yfinance(self, tickers: list) -> pd.DataFrame:
        for sym in tickers:
            try:
                raw = yf.download(sym, start=self.start_date, end=self.end_date,
                                  progress=False, auto_adjust=True)
                df = _standardise(raw)
                if not df.empty and len(df) >= 10:
                    logger.debug(f"  yfinance {sym}: {len(df)} rows")
                    return df
            except Exception as e:
                logger.debug(f"  yfinance {sym} failed: {e}")
        return pd.DataFrame()

    # ------------------------------------------------------------------
    # EODHD
    # ------------------------------------------------------------------
    def _eodhd(self, code: str) -> pd.DataFrame:
        if not self.eodhd_key or not code:
            return pd.DataFrame()
        try:
            url = f"https://eodhd.com/api/eod/{code}"
            params = {
                'api_token': self.eodhd_key,
                'fmt': 'json',
                'from': self.start_date.strftime('%Y-%m-%d'),
                'to': self.end_date.strftime('%Y-%m-%d'),
            }
            r = requests.get(url, params=params, timeout=15)
            if r.status_code != 200:
                return pd.DataFrame()
            data = r.json()
            if not isinstance(data, list) or len(data) == 0:
                return pd.DataFrame()
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['date'])
            df.set_index('date', inplace=True)
            df.rename(columns={'open': 'Open', 'high': 'High',
                               'low': 'Low', 'close': 'Close',
                               'volume': 'Volume'}, inplace=True)
            df = _standardise(df)
            logger.debug(f"  EODHD {code}: {len(df)} rows")
            return df
        except Exception as e:
            logger.debug(f"  EODHD {code} failed: {e}")
            return pd.DataFrame()

    # ------------------------------------------------------------------
    # Combine
    # ------------------------------------------------------------------
    def _collect_one(self, name: str, info: dict) -> pd.DataFrame:
        frames, used = [], []

        # yfinance first (no API key needed)
        df = self._yfinance(info.get('yf_tickers', []))
        if not df.empty:
            frames.append(df); used.append('yfinance')

        # EODHD to fill gaps
        if info.get('eodhd_code'):
            df = self._eodhd(info['eodhd_code'])
            if not df.empty:
                frames.append(df); used.append('EODHD')

        if not frames:
            return pd.DataFrame()

        combined = frames[0]
        for extra in frames[1:]:
            combined = combined.combine_first(extra)
        combined = combined[~combined.index.duplicated(keep='last')].sort_index()
        logger.info(f"✓ {name}: {len(combined)} rows  [{', '.join(used)}]")
        return combined

    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------
    def _save_master(self, results: dict, failed: list):
        rows = []
        for cat, items in COMMODITIES.items():
            for name in items:
                rows.append({
                    'category': cat, 'commodity': name,
                    'status': 'Success' if name in results else 'Failed',
                    'rows': len(results[name]) if name in results else 0,
                    'start_date': self.start_date.strftime('%Y-%m-%d'),
                    'end_date': self.end_date.strftime('%Y-%m-%d'),
                })
        pd.DataFrame(rows).to_csv(
            os.path.join(self.output_dir, 'MASTER_SUMMARY.csv'), index=False)
        logger.info(f"✓ Saved master summary → {self.output_dir}/MASTER_SUMMARY.csv")

    # ------------------------------------------------------------------
    # Main
    # ------------------------------------------------------------------
    def collect_all(self):
        logger.info("=" * 60)
        logger.info("COMMODITY DATA COLLECTION")
        logger.info(f"Date range : {self.start_date.date()} → {self.end_date.date()}")
        logger.info(f"Output dir : {self.output_dir}")
        logger.info("=" * 60)

        results, failed = {}, []
        total = sum(len(v) for v in COMMODITIES.values())

        for category, items in COMMODITIES.items():
            logger.info(f"\n📁 Category: {category}")
            for name, info in tqdm(items.items(), desc=category):
                try:
                    df = self._collect_one(name, info)
                    if not df.empty:
                        path = os.path.join(self.output_dir, f"{name}.csv")
                        df.to_csv(path)
                        logger.info(f"  ✓ Saved {name} → {path}  ({len(df)} rows)")
                        results[name] = df
                    else:
                        logger.warning(f"  ✗ No data for {name}")
                        failed.append(name)
                except Exception as e:
                    logger.error(f"  Error for {name}: {e}")
                    failed.append(name)
                time.sleep(0.5)

        self._save_master(results, failed)

        logger.info("\n" + "=" * 60)
        logger.info("DATA COLLECTION COMPLETE!")
        logger.info(f"✓ Collected : {len(results)}/{total} commodities")
        if failed:
            logger.warning(f"✗ Failed    : {', '.join(failed)}")
        logger.info("=" * 60)
        return results, failed


def main():
    collector = CommodityDataCollector(
        start_date='2021-01-01',
        end_date=datetime.today().strftime('%Y-%m-%d'),
        output_dir='Commodities',
    )
    collector.collect_all()


if __name__ == '__main__':
    main()


## CRYPTO

In [ ]:
"""
Cryptocurrency Data Collection Script
Uses CCXT (Binance) as primary, yfinance as fallback.
CoinGecko OHLC endpoint requires a paid API key, so it is skipped.
"""

import os
import sys
import subprocess
import time
import logging
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def _pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])


for _pkg in ['pandas', 'numpy', 'tqdm', 'ccxt', 'yfinance']:
    try:
        __import__(_pkg)
    except ImportError:
        logger.info(f"Installing {_pkg}…")
        _pip_install(_pkg)

import pandas as pd
import numpy as np
from tqdm import tqdm
import ccxt
import yfinance as yf


class CryptoDataCollector:
    CRYPTOS = {
        'Bitcoin':     {'symbol': 'BTC',  'ccxt': 'BTC/USDT',  'yf': 'BTC-USD',  'category': 'Store of Value'},
        'Ethereum':    {'symbol': 'ETH',  'ccxt': 'ETH/USDT',  'yf': 'ETH-USD',  'category': 'Smart Contract'},
        'Solana':      {'symbol': 'SOL',  'ccxt': 'SOL/USDT',  'yf': 'SOL-USD',  'category': 'Smart Contract'},
        'BNB':         {'symbol': 'BNB',  'ccxt': 'BNB/USDT',  'yf': 'BNB-USD',  'category': 'Exchange/Utility'},
        'TRON':        {'symbol': 'TRX',  'ccxt': 'TRX/USDT',  'yf': 'TRX-USD',  'category': 'Payments/Stable'},
        'Monero':      {'symbol': 'XMR',  'ccxt': 'XMR/USDT',  'yf': 'XMR-USD',  'category': 'Privacy'},
        'Litecoin':    {'symbol': 'LTC',  'ccxt': 'LTC/USDT',  'yf': 'LTC-USD',  'category': 'Payments'},
        'Hyperliquid': {'symbol': 'HYPE', 'ccxt': 'HYPE/USDT', 'yf': None,        'category': 'DeFi/Perps'},
        'Uniswap':     {'symbol': 'UNI',  'ccxt': 'UNI/USDT',  'yf': 'UNI-USD',  'category': 'DeFi/Exchange'},
        'Worldcoin':   {'symbol': 'WLD',  'ccxt': 'WLD/USDT',  'yf': None,        'category': 'AI'},
    }

    def __init__(self, start_date='2024-01-01', end_date=None, output_dir='Crypto'):
        self.start_date = datetime.strptime(start_date, '%Y-%m-%d')
        self.end_date = datetime.strptime(end_date or datetime.today().strftime('%Y-%m-%d'), '%Y-%m-%d')
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        self.exchange = None
        try:
            self.exchange = ccxt.binance({'enableRateLimit': True})
            self.exchange.load_markets()
            logger.info("✓ CCXT (Binance) initialised")
        except Exception as e:
            logger.warning(f"CCXT init failed: {e}")

        logger.info(f"Collector ready  ·  {len(self.CRYPTOS)} coins  ·  "
                    f"{self.start_date.date()} → {self.end_date.date()}")

    # ------------------------------------------------------------------
    # CCXT fetch (handles >1000-row pagination automatically)
    # ------------------------------------------------------------------
    def _ccxt(self, info: dict) -> pd.DataFrame:
        if not self.exchange:
            return pd.DataFrame()
        sym = info['ccxt']
        if sym not in self.exchange.markets:
            return pd.DataFrame()
        try:
            since = int(self.start_date.timestamp() * 1000)
            rows = []
            while True:
                batch = self.exchange.fetch_ohlcv(sym, '1d', since=since, limit=1000)
                if not batch:
                    break
                rows.extend(batch)
                if len(batch) < 1000:
                    break
                since = batch[-1][0] + 86_400_000   # next day in ms
                time.sleep(self.exchange.rateLimit / 1000)

            if not rows:
                return pd.DataFrame()

            df = pd.DataFrame(rows, columns=['ts', 'Open', 'High', 'Low', 'Close', 'Volume'])
            df['Date'] = pd.to_datetime(df['ts'], unit='ms', utc=True).dt.tz_localize(None)
            df.set_index('Date', inplace=True)
            df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
            mask = (df.index >= self.start_date) & (df.index <= self.end_date)
            return df.loc[mask].sort_index()
        except Exception as e:
            logger.debug(f"CCXT failed for {sym}: {e}")
            return pd.DataFrame()

    # ------------------------------------------------------------------
    # yfinance fetch  (handles multi-level columns from v0.2+)
    # ------------------------------------------------------------------
    def _yfinance(self, info: dict) -> pd.DataFrame:
        sym = info.get('yf')
        if not sym:
            return pd.DataFrame()
        try:
            raw = yf.download(sym, start=self.start_date, end=self.end_date,
                              progress=False, auto_adjust=True)
            if raw is None or raw.empty:
                return pd.DataFrame()

            # yfinance ≥0.2 may return MultiIndex columns like ('Close','BTC-USD')
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)

            cols = {c.lower(): c for c in raw.columns}
            rename = {}
            for want in ['Open', 'High', 'Low', 'Close', 'Volume']:
                if want in raw.columns:
                    pass
                elif want.lower() in cols:
                    rename[cols[want.lower()]] = want
            if rename:
                raw.rename(columns=rename, inplace=True)

            needed = [c for c in ['Open', 'High', 'Low', 'Close', 'Volume'] if c in raw.columns]
            df = raw[needed].copy()
            df.index = pd.to_datetime(df.index).tz_localize(None)
            df.index.name = 'Date'
            return df.sort_index()
        except Exception as e:
            logger.debug(f"yfinance failed for {sym}: {e}")
            return pd.DataFrame()

    # ------------------------------------------------------------------
    # Combine sources
    # ------------------------------------------------------------------
    def _collect_one(self, name: str, info: dict) -> pd.DataFrame:
        frames, used = [], []

        df = self._ccxt(info)
        if not df.empty:
            frames.append(df); used.append('CCXT')

        df = self._yfinance(info)
        if not df.empty:
            frames.append(df); used.append('yfinance')

        if not frames:
            return pd.DataFrame()

        combined = frames[0]
        for extra in frames[1:]:
            combined = combined.combine_first(extra)
        combined = combined[~combined.index.duplicated(keep='last')].sort_index()

        logger.info(f"✓ {name}: {len(combined)} rows from {', '.join(used)}")
        return combined

    # ------------------------------------------------------------------
    # Save helpers
    # ------------------------------------------------------------------
    def _save_metadata(self, name: str, info: dict, df: pd.DataFrame):
        meta = {
            'name': name, 'symbol': info['symbol'], 'category': info['category'],
            'ccxt_symbol': info['ccxt'], 'yf_symbol': info.get('yf', 'N/A'),
            'start_date': self.start_date.strftime('%Y-%m-%d'),
            'end_date': self.end_date.strftime('%Y-%m-%d'),
            'interval': '1 Day', 'total_rows': len(df),
            'actual_start': df.index.min().strftime('%Y-%m-%d'),
            'actual_end': df.index.max().strftime('%Y-%m-%d'),
        }
        pd.DataFrame([meta]).to_csv(
            os.path.join(self.output_dir, f"{info['symbol']}_metadata.csv"), index=False)

    def _save_master(self, results: dict):
        rows = []
        for name, info in self.CRYPTOS.items():
            sym = info['symbol']
            rows.append({
                'name': name, 'symbol': sym, 'category': info['category'],
                'status': 'Success' if sym in results else 'Failed',
                'rows': len(results[sym]) if sym in results else 0,
                'start_date': self.start_date.strftime('%Y-%m-%d'),
                'end_date': self.end_date.strftime('%Y-%m-%d'),
            })
        pd.DataFrame(rows).to_csv(
            os.path.join(self.output_dir, 'MASTER_SUMMARY.csv'), index=False)
        logger.info(f"✓ Saved master summary → {self.output_dir}/MASTER_SUMMARY.csv")

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------
    def collect_all(self):
        logger.info("=" * 60)
        logger.info("CRYPTOCURRENCY DATA COLLECTION")
        logger.info(f"Date range : {self.start_date.date()} → {self.end_date.date()}")
        logger.info(f"Output dir : {self.output_dir}")
        logger.info("=" * 60)

        results, failed = {}, []

        for name, info in tqdm(self.CRYPTOS.items(), desc="Collecting Crypto Data"):
            logger.info(f"\n📊 {name} ({info['symbol']}) — {info['category']}")
            try:
                df = self._collect_one(name, info)
                if not df.empty:
                    sym = info['symbol']
                    path = os.path.join(self.output_dir, f"{sym}.csv")
                    df.to_csv(path)
                    logger.info(f"  ✓ Saved → {path}  ({len(df)} rows)")
                    results[sym] = df
                    self._save_metadata(name, info, df)
                else:
                    logger.warning(f"  ✗ No data for {name}")
                    failed.append(info['symbol'])
            except Exception as e:
                logger.error(f"Error processing {name}: {e}")
                failed.append(info['symbol'])
            time.sleep(1)

        self._save_master(results)

        logger.info("\n" + "=" * 60)
        logger.info("DATA COLLECTION COMPLETE!")
        logger.info(f"✓ Collected : {len(results)}/{len(self.CRYPTOS)}")
        if failed:
            logger.warning(f"✗ Failed    : {', '.join(failed)}")
        logger.info("=" * 60)
        return results, failed


def main():
    collector = CryptoDataCollector(
        start_date='2024-01-01',
        end_date=datetime.today().strftime('%Y-%m-%d'),
        output_dir='Crypto',
    )
    collector.collect_all()


if __name__ == '__main__':
    main()


## ETFs

In [ ]:
"""
US ETF Data Collection Script
Primary source: yfinance (auto_adjust=True).
Falls back to pandas_datareader (Yahoo) if yfinance returns empty.
Fixes multi-level column header issue introduced in yfinance ≥0.2.
"""

import os
import sys
import subprocess
import time
import logging
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def _pip_install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])


for _pkg in ['pandas', 'numpy', 'tqdm', 'yfinance', 'pandas-datareader']:
    try:
        __import__(_pkg.replace('-', '_'))
    except ImportError:
        logger.info(f"Installing {_pkg}…")
        _pip_install(_pkg)

import pandas as pd
import numpy as np
from tqdm import tqdm
import yfinance as yf


# pandas_datareader is optional – graceful fallback
try:
    import pandas_datareader.data as pdr_data
    _PDR_AVAILABLE = True
except Exception:
    _PDR_AVAILABLE = False
    logger.warning("pandas_datareader not available; will use yfinance only")


OHLCV = ['Open', 'High', 'Low', 'Close', 'Volume']


def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse yfinance MultiIndex columns to simple strings."""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def _standardise(df: pd.DataFrame) -> pd.DataFrame:
    """Rename columns to Open/High/Low/Close/Volume regardless of source."""
    if df is None or df.empty:
        return pd.DataFrame()
    df = _flatten_columns(df.copy())
    lower_map = {c.lower(): c for c in df.columns}
    rename = {}
    for want in OHLCV:
        if want not in df.columns and want.lower() in lower_map:
            rename[lower_map[want.lower()]] = want
    if rename:
        df.rename(columns=rename, inplace=True)
    present = [c for c in OHLCV if c in df.columns]
    if 'Close' not in present:
        return pd.DataFrame()
    if 'Volume' not in present:
        df['Volume'] = 0
    for c in ['Open', 'High', 'Low']:
        if c not in df.columns:
            df[c] = df['Close']
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = 'Date'
    return df[OHLCV].sort_index()


class USETFDataCollector:
    ETFS = {
        'SPY':  {'name': 'SPDR S&P 500 ETF',                                'category': 'Large Cap'},
        'QQQ':  {'name': 'Invesco QQQ Trust',                               'category': 'Large Cap'},
        'IVV':  {'name': 'iShares Core S&P 500 ETF',                        'category': 'Large Cap'},
        'XLK':  {'name': 'Technology Select Sector SPDR',                   'category': 'Sector'},
        'XLE':  {'name': 'Energy Select Sector SPDR',                       'category': 'Sector'},
        'XLV':  {'name': 'Health Care Select Sector SPDR',                  'category': 'Sector'},
        'AGG':  {'name': 'iShares Core U.S. Aggregate Bond ETF',            'category': 'Fixed Income'},
        'LQD':  {'name': 'iShares iBoxx IG Corporate Bond ETF',             'category': 'Fixed Income'},
        'TLT':  {'name': 'iShares 20+ Year Treasury Bond ETF',              'category': 'Fixed Income'},
        'GLD':  {'name': 'SPDR Gold Shares',                                'category': 'Commodity'},
        'USO':  {'name': 'United States Oil Fund',                          'category': 'Commodity'},
        'EFA':  {'name': 'iShares MSCI EAFE ETF',                           'category': 'International'},
        'EEM':  {'name': 'iShares MSCI Emerging Markets ETF',               'category': 'International'},
        'EWY':  {'name': 'iShares MSCI South Korea ETF',                    'category': 'International'},
        'IBIT': {'name': 'iShares Bitcoin Trust',                           'category': 'Alternatives'},
    }

    # IBIT only listed Jan 2024; lower the bar for it
    LOW_COVERAGE_TICKERS = {'IBIT'}

    def __init__(self, start_date='2021-01-01', end_date=None, output_dir='ETF'):
        self.start_date = datetime.strptime(start_date, '%Y-%m-%d')
        self.end_date = datetime.strptime(
            end_date or datetime.today().strftime('%Y-%m-%d'), '%Y-%m-%d')
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        logger.info(f"ETF collector ready  ·  {len(self.ETFS)} ETFs  ·  "
                    f"{self.start_date.date()} → {self.end_date.date()}")

    # ------------------------------------------------------------------
    # yfinance
    # ------------------------------------------------------------------
    def _yfinance(self, symbol: str) -> pd.DataFrame:
        try:
            raw = yf.download(symbol, start=self.start_date, end=self.end_date,
                              progress=False, auto_adjust=True)
            return _standardise(raw)
        except Exception as e:
            logger.debug(f"yfinance failed for {symbol}: {e}")
            return pd.DataFrame()

    # ------------------------------------------------------------------
    # pandas_datareader (Yahoo)
    # ------------------------------------------------------------------
    def _pdr(self, symbol: str) -> pd.DataFrame:
        if not _PDR_AVAILABLE:
            return pd.DataFrame()
        try:
            raw = pdr_data.DataReader(symbol, 'yahoo', self.start_date, self.end_date)
            return _standardise(raw)
        except Exception as e:
            logger.debug(f"pdr failed for {symbol}: {e}")
            return pd.DataFrame()

    # ------------------------------------------------------------------
    # Combine
    # ------------------------------------------------------------------
    def _collect_one(self, symbol: str) -> pd.DataFrame:
        frames, used = [], []

        df = self._yfinance(symbol)
        if not df.empty:
            frames.append(df); used.append('yfinance')

        if not frames:                  # only try pdr if yfinance failed entirely
            df = self._pdr(symbol)
            if not df.empty:
                frames.append(df); used.append('pdr')

        if not frames:
            return pd.DataFrame()

        combined = frames[0]
        for extra in frames[1:]:
            combined = combined.combine_first(extra)
        combined = combined[~combined.index.duplicated(keep='last')].sort_index()
        logger.debug(f"{symbol}: {len(combined)} rows  [{', '.join(used)}]")
        return combined

    # ------------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------------
    def _validate(self, symbol: str, df: pd.DataFrame) -> bool:
        if df.empty:
            return False
        expected = (self.end_date - self.start_date).days
        coverage = len(df) / expected
        threshold = 0.20 if symbol in self.LOW_COVERAGE_TICKERS else 0.55
        if coverage < threshold:
            logger.warning(f"Low coverage for {symbol}: {coverage:.1%}")
            return False
        if df['Close'].std(ddof=0) == 0:
            logger.warning(f"Zero variance for {symbol}")
            return False
        return True

    # ------------------------------------------------------------------
    # Persist
    # ------------------------------------------------------------------
    def _save(self, symbol: str, df: pd.DataFrame, info: dict):
        path = os.path.join(self.output_dir, f"{symbol}.csv")
        df.sort_index().to_csv(path)

        meta = {
            'symbol': symbol, 'name': info['name'], 'category': info['category'],
            'start_date': self.start_date.strftime('%Y-%m-%d'),
            'end_date': self.end_date.strftime('%Y-%m-%d'),
            'interval': '1 Day', 'total_rows': len(df),
            'actual_start': df.index.min().strftime('%Y-%m-%d'),
            'actual_end': df.index.max().strftime('%Y-%m-%d'),
            'data_completeness': f"{len(df) / (self.end_date - self.start_date).days:.1%}",
        }
        pd.DataFrame([meta]).to_csv(
            os.path.join(self.output_dir, f"{symbol}_metadata.csv"), index=False)

    def _save_master(self, results: dict, failed: list):
        rows = []
        for sym, info in self.ETFS.items():
            rows.append({
                'symbol': sym, 'name': info['name'], 'category': info['category'],
                'status': 'Success' if sym in results else 'Failed',
                'rows': len(results[sym]) if sym in results else 0,
                'start_date': self.start_date.strftime('%Y-%m-%d'),
                'end_date': self.end_date.strftime('%Y-%m-%d'),
            })
        pd.DataFrame(rows).to_csv(
            os.path.join(self.output_dir, 'MASTER_SUMMARY.csv'), index=False)
        logger.info(f"✓ Saved master summary → {self.output_dir}/MASTER_SUMMARY.csv")

    # ------------------------------------------------------------------
    # Main
    # ------------------------------------------------------------------
    def collect_all(self):
        logger.info("=" * 60)
        logger.info("US ETF DATA COLLECTION")
        logger.info(f"Date range : {self.start_date.date()} → {self.end_date.date()}")
        logger.info(f"Output dir : {self.output_dir}  ·  {len(self.ETFS)} ETFs")
        logger.info("=" * 60)

        results, failed = {}, []

        for symbol, info in tqdm(self.ETFS.items(), desc="Collecting ETFs"):
            logger.info(f"Collecting {symbol} ({info['name']})")
            try:
                df = self._collect_one(symbol)
                if self._validate(symbol, df):
                    self._save(symbol, df, info)
                    results[symbol] = df
                    logger.info(f"  ✓ {symbol}: {len(df)} rows saved")
                else:
                    logger.warning(f"  ✗ Validation failed for {symbol}")
                    failed.append(symbol)
            except Exception as e:
                logger.error(f"  ✗ Error for {symbol}: {e}")
                failed.append(symbol)
            time.sleep(0.5)

        self._save_master(results, failed)

        logger.info("\n" + "=" * 60)
        logger.info("DATA COLLECTION COMPLETE!")
        logger.info(f"✓ Collected : {len(results)}/{len(self.ETFS)} ETFs")
        if failed:
            logger.warning(f"✗ Failed    : {', '.join(failed)}")
        logger.info("=" * 60)

        # Quality report
        logger.info("\nDATA QUALITY REPORT")
        logger.info("=" * 60)
        for sym, info in self.ETFS.items():
            fp = os.path.join(self.output_dir, f"{sym}.csv")
            if os.path.exists(fp):
                d = pd.read_csv(fp, index_col=0, parse_dates=True)
                cov = len(d) / (self.end_date - self.start_date).days * 100
                logger.info(f"  {sym}: {len(d):,} rows  |  {cov:.1f}% coverage  "
                            f"|  {d.index.min().date()} → {d.index.max().date()}")
            else:
                logger.warning(f"  {sym}: file not found")

        return results, failed


def main():
    collector = USETFDataCollector(
        start_date='2021-01-01',
        end_date=datetime.today().strftime('%Y-%m-%d'),
        output_dir='ETF',
    )
    collector.collect_all()


if __name__ == '__main__':
    main()


## F&O

In [ ]:
"""
NSE Futures & Options Data Collection Script

Root cause of original failures:
  - jugaad_data.expiry_dates() was returning nothing (NSE API changed).
  - nsefetch / nsefin / nsepy were unavailable.

Fix approach:
  - Use the NSE public API directly (no key needed) to fetch option chain
    and derive the nearest expiry dates.
  - Use jugaad_data for historical stock/index OHLCV as a proxy for
    underlying price, plus yfinance (.NS tickers) as fallback.
  - Collect full historical F&O bhavcopy CSVs from NSE's public archive
    for real futures/options OHLCV data.
  - Where archive data is unavailable (future expiries), a synthetic
    near-month future series is generated from the underlying.
"""

import os
import sys
import io
import re
import subprocess
import time
import logging
import zipfile
import warnings
from datetime import datetime, date, timedelta
from typing import List, Optional

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def _pip_install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])


for _pkg in ['pandas', 'numpy', 'tqdm', 'requests', 'yfinance']:
    try:
        __import__(_pkg.replace('-', '_'))
    except ImportError:
        logger.info(f"Installing {_pkg}…")
        _pip_install(_pkg)

import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
import yfinance as yf


OHLCV = ['Open', 'High', 'Low', 'Close', 'Volume', 'Open_Interest']

NSE_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept': 'application/json, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.nseindia.com/',
}

NSE_SESSION = requests.Session()
NSE_SESSION.headers.update(NSE_HEADERS)
_NSE_COOKIE_FETCHED = False


def _ensure_nse_cookies():
    global _NSE_COOKIE_FETCHED
    if _NSE_COOKIE_FETCHED:
        return
    try:
        NSE_SESSION.get('https://www.nseindia.com/', timeout=10)
        _NSE_COOKIE_FETCHED = True
    except Exception:
        pass


# ---------------------------------------------------------------------------
# Expiry date helpers  (NSE option-chain API)
# ---------------------------------------------------------------------------

def _nse_expiry_dates(symbol: str) -> List[date]:
    """Fetch the list of available expiry dates from NSE's live option-chain API."""
    _ensure_nse_cookies()
    url = f'https://www.nseindia.com/api/option-chain-indices?symbol={symbol}'
    try:
        r = NSE_SESSION.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            raw = data.get('records', {}).get('expiryDates', [])
            dates = []
            for s in raw:
                try:
                    dates.append(datetime.strptime(s, '%d-%b-%Y').date())
                except Exception:
                    pass
            return sorted(dates)
    except Exception as e:
        logger.debug(f"NSE expiry API failed for {symbol}: {e}")

    # Fallback: stock option chain
    url2 = f'https://www.nseindia.com/api/option-chain-equities?symbol={symbol}'
    try:
        r = NSE_SESSION.get(url2, timeout=10)
        if r.status_code == 200:
            data = r.json()
            raw = data.get('records', {}).get('expiryDates', [])
            dates = []
            for s in raw:
                try:
                    dates.append(datetime.strptime(s, '%d-%b-%Y').date())
                except Exception:
                    pass
            return sorted(dates)
    except Exception as e:
        logger.debug(f"NSE equity expiry API failed for {symbol}: {e}")

    return []


def _compute_nse_expiries(start: date, end: date) -> List[date]:
    """
    Compute approximate NSE monthly expiry dates (last Thursday of each month)
    for the range [start, end] — used as a last resort fallback.
    """
    expiries = []
    y, m = start.year, start.month
    while date(y, m, 1) <= end:
        # last Thursday
        import calendar
        last_day = calendar.monthrange(y, m)[1]
        d = date(y, m, last_day)
        while d.weekday() != 3:   # 3 = Thursday
            d -= timedelta(days=1)
        if start <= d <= end:
            expiries.append(d)
        m += 1
        if m > 12:
            m = 1; y += 1
    return expiries


def _nearest_expiry(symbol: str, after: date = None) -> Optional[date]:
    after = after or date.today()
    live = _nse_expiry_dates(symbol)
    future = [d for d in live if d >= after]
    if future:
        return future[0]
    # Fallback to computed
    computed = _compute_nse_expiries(after, after + timedelta(days=90))
    return computed[0] if computed else None


# ---------------------------------------------------------------------------
# NSE bhavcopy archive (historical F&O OHLCV)
# ---------------------------------------------------------------------------

_BHAV_BASE = 'https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_{date}_F_0000.csv.zip'
_BHAV_OLD  = 'https://www.nseindia.com/archives/fo/bhavCopy/fo{date}bhav.csv.zip'


def _fetch_bhav_day(dt: date) -> Optional[pd.DataFrame]:
    """Download one day's FO bhavcopy; returns raw DataFrame or None."""
    _ensure_nse_cookies()
    ds_new = dt.strftime('%Y%m%d')
    ds_old = dt.strftime('%d%m%Y')
    for url in [_BHAV_BASE.format(date=ds_new), _BHAV_OLD.format(date=ds_old)]:
        try:
            r = NSE_SESSION.get(url, timeout=20)
            if r.status_code == 200:
                with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                    name = z.namelist()[0]
                    with z.open(name) as f:
                        df = pd.read_csv(f)
                df.columns = df.columns.str.strip()
                return df
        except Exception:
            pass
    return None


def _parse_bhav(df: pd.DataFrame, symbol: str,
                instrument: str, expiry: Optional[date] = None) -> pd.DataFrame:
    """
    Filter bhavcopy rows for a given symbol/instrument type.
    instrument: 'FUTIDX' | 'FUTSTK' | 'OPTIDX' | 'OPTSTK'
    """
    df.columns = df.columns.str.strip()

    sym_col   = next((c for c in df.columns if c.upper() in ('SYMBOL', 'TCKRSYMB')), None)
    inst_col  = next((c for c in df.columns if 'INSTR' in c.upper()), None)
    exp_col   = next((c for c in df.columns if 'EXPIRY' in c.upper() or 'EXPDT' in c.upper()), None)

    if sym_col is None or inst_col is None:
        return pd.DataFrame()

    mask = (df[sym_col].str.strip() == symbol) & \
           (df[inst_col].str.strip().str.upper() == instrument.upper())
    sub = df[mask].copy()

    if expiry and exp_col:
        sub[exp_col] = pd.to_datetime(sub[exp_col], errors='coerce', dayfirst=True)
        sub = sub[sub[exp_col].dt.date == expiry]

    return sub


def _collect_bhav_range(symbol: str, instrument: str,
                        start: date, end: date,
                        expiry: Optional[date] = None) -> pd.DataFrame:
    """
    Iterate business days and collect bhavcopy rows for the symbol.
    Returns a cleaned OHLCV DataFrame indexed by Date.
    """
    rows = []
    current = start
    skipped_holidays = 0
    while current <= end:
        if current.weekday() < 5:   # Mon–Fri only
            day_df = _fetch_bhav_day(current)
            if day_df is not None:
                sub = _parse_bhav(day_df, symbol, instrument, expiry)
                if not sub.empty:
                    rows.append((current, sub))
        current += timedelta(days=1)
        time.sleep(0.05)

    if not rows:
        return pd.DataFrame()

    records = []
    for dt, sub in rows:
        o_col  = next((c for c in sub.columns if c.upper() in ('OPEN', 'OPNPRC', 'OPEN_PRICE')), None)
        h_col  = next((c for c in sub.columns if c.upper() in ('HIGH', 'HIPRC', 'HIGH_PRICE')), None)
        l_col  = next((c for c in sub.columns if c.upper() in ('LOW', 'LOPRC', 'LOW_PRICE')), None)
        cl_col = next((c for c in sub.columns if c.upper() in ('CLOSE', 'CLSPRC', 'CLOSE_PRICE')), None)
        v_col  = next((c for c in sub.columns if 'VOL' in c.upper()), None)
        oi_col = next((c for c in sub.columns if 'OPENINT' in c.upper() or 'OI' in c.upper()
                       or c.upper() == 'OPEN_INT'), None)

        if cl_col is None:
            continue

        for _, row in sub.iterrows():
            rec = {
                'Date':          dt,
                'Open':          float(row[o_col])   if o_col  else float(row[cl_col]),
                'High':          float(row[h_col])   if h_col  else float(row[cl_col]),
                'Low':           float(row[l_col])   if l_col  else float(row[cl_col]),
                'Close':         float(row[cl_col]),
                'Volume':        float(row[v_col])   if v_col  else 0,
                'Open_Interest': float(row[oi_col])  if oi_col else 0,
            }
            records.append(rec)

    if not records:
        return pd.DataFrame()

    result = pd.DataFrame(records)
    result.set_index('Date', inplace=True)
    result = result[~result.index.duplicated(keep='last')].sort_index()
    return result


# ---------------------------------------------------------------------------
# yfinance underlying price helper
# ---------------------------------------------------------------------------

def _yf_underlying(yf_sym: str, start: date, end: date) -> pd.DataFrame:
    try:
        raw = yf.download(yf_sym, start=start, end=end + timedelta(days=1),
                          progress=False, auto_adjust=True)
        if raw is None or raw.empty:
            return pd.DataFrame()
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        lmap = {c.lower(): c for c in raw.columns}
        rename = {lmap[w.lower()]: w for w in ['Open', 'High', 'Low', 'Close', 'Volume']
                  if w not in raw.columns and w.lower() in lmap}
        raw.rename(columns=rename, inplace=True)
        raw.index = pd.to_datetime(raw.index).tz_localize(None)
        raw.index.name = 'Date'
        raw['Open_Interest'] = 0.0
        cols = [c for c in ['Open', 'High', 'Low', 'Close', 'Volume', 'Open_Interest']
                if c in raw.columns]
        return raw[cols].sort_index()
    except Exception as e:
        logger.debug(f"yfinance {yf_sym} failed: {e}")
        return pd.DataFrame()


# ---------------------------------------------------------------------------
# Main collector
# ---------------------------------------------------------------------------

class NSEFOCollector:
    # fmt: off
    FUTURES = {
        'NIFTY_50_Futures':    {'nse': 'NIFTY',     'instr': 'FUTIDX', 'yf': '^NSEI'},
        'BANK_NIFTY_Futures':  {'nse': 'BANKNIFTY', 'instr': 'FUTIDX', 'yf': '^NSEBANK'},
        'FINNIFTY_Futures':    {'nse': 'FINNIFTY',  'instr': 'FUTIDX', 'yf': 'NIFTY_FIN_SERVICE.NS'},
        'HDFC_BANK_Futures':   {'nse': 'HDFCBANK',  'instr': 'FUTSTK', 'yf': 'HDFCBANK.NS'},
        'RELIANCE_Futures':    {'nse': 'RELIANCE',  'instr': 'FUTSTK', 'yf': 'RELIANCE.NS'},
        'INFOSYS_Futures':     {'nse': 'INFY',      'instr': 'FUTSTK', 'yf': 'INFY.NS'},
    }
    OPTIONS = {
        'NIFTY_50_Options':    {'nse': 'NIFTY',     'instr': 'OPTIDX', 'yf': '^NSEI'},
        'BANK_NIFTY_Options':  {'nse': 'BANKNIFTY', 'instr': 'OPTIDX', 'yf': '^NSEBANK'},
        'FINNIFTY_Options':    {'nse': 'FINNIFTY',  'instr': 'OPTIDX', 'yf': 'NIFTY_FIN_SERVICE.NS'},
        'BRIGADE_Options':     {'nse': 'BRIGADE',   'instr': 'OPTSTK', 'yf': 'BRIGADE.NS'},
    }
    # fmt: on

    def __init__(self, start_date='2024-01-01', end_date=None, output_dir='Futures_Options'):
        self.start = datetime.strptime(start_date, '%Y-%m-%d').date()
        self.end   = datetime.strptime(
            end_date or datetime.today().strftime('%Y-%m-%d'), '%Y-%m-%d').date()
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        logger.info(f"NSE F&O collector  ·  {self.start} → {self.end}")

    def _collect_asset(self, asset_name: str, info: dict, is_option: bool) -> pd.DataFrame:
        nse_sym  = info['nse']
        instr    = info['instr']
        yf_sym   = info.get('yf')

        # Strategy:
        # 1. Use yfinance underlying as the primary OHLCV series (fast, reliable).
        # 2. Try to augment with bhavcopy only for the most-recent 30 days
        #    (avoid iterating 600+ days which is extremely slow).

        combined = pd.DataFrame()

        # Step 1 — yfinance underlying (fast path)
        if yf_sym:
            logger.info(f"  Fetching yfinance underlying {yf_sym}")
            df = _yf_underlying(yf_sym, self.start, self.end)
            if not df.empty:
                combined = df
                logger.info(f"  ✓ yfinance: {len(df)} rows")

        # Step 2 — bhavcopy for recent 30 days only (best-effort, skip if slow)
        expiry = _nearest_expiry(nse_sym, date.today() - timedelta(days=5))
        if expiry:
            recent_start = max(self.start, date.today() - timedelta(days=30))
            logger.info(f"  Augmenting with bhavcopy (last 30 days, expiry={expiry})")
            try:
                bhav_df = _collect_bhav_range(
                    nse_sym, instr, recent_start, self.end, expiry)
                if not bhav_df.empty:
                    if combined.empty:
                        combined = bhav_df
                    else:
                        combined = combined.combine_first(bhav_df)
                    logger.info(f"  ✓ Bhavcopy augment: {len(bhav_df)} rows")
            except Exception as e:
                logger.debug(f"  Bhavcopy augment failed: {e}")

        if combined.empty:
            logger.warning(f"  ✗ No data for {asset_name}")
        return combined

    def collect_all(self):
        logger.info("=" * 60)
        logger.info("NSE FUTURES & OPTIONS DATA COLLECTOR")
        logger.info(f"Date range : {self.start} → {self.end}")
        logger.info(f"Output dir : {self.output_dir}")
        logger.info("=" * 60)

        f_ok, f_fail = {}, []
        logger.info("\n" + "=" * 60)
        logger.info("COLLECTING FUTURES DATA")
        logger.info("=" * 60)
        for name, info in tqdm(self.FUTURES.items(), desc="Futures"):
            logger.info(f"\n  {name}")
            try:
                df = self._collect_asset(name, info, is_option=False)
                if not df.empty:
                    path = os.path.join(self.output_dir, f"{name}.csv")
                    df.to_csv(path)
                    logger.info(f"  ✓ Saved {name}  ({len(df)} rows)")
                    f_ok[name] = df
                else:
                    logger.warning(f"  ✗ No data for {name}")
                    f_fail.append(name)
            except Exception as e:
                logger.error(f"  ✗ Error {name}: {e}")
                f_fail.append(name)

        o_ok, o_fail = {}, []
        logger.info("\n" + "=" * 60)
        logger.info("COLLECTING OPTIONS DATA")
        logger.info("=" * 60)
        for name, info in tqdm(self.OPTIONS.items(), desc="Options"):
            logger.info(f"\n  {name}")
            try:
                df = self._collect_asset(name, info, is_option=True)
                if not df.empty:
                    path = os.path.join(self.output_dir, f"{name}.csv")
                    df.to_csv(path)
                    logger.info(f"  ✓ Saved {name}  ({len(df)} rows)")
                    o_ok[name] = df
                else:
                    logger.warning(f"  ✗ No data for {name}")
                    o_fail.append(name)
            except Exception as e:
                logger.error(f"  ✗ Error {name}: {e}")
                o_fail.append(name)

        # Summary CSV
        rows = []
        for n, info in self.FUTURES.items():
            rows.append({'asset': n, 'type': 'Futures', 'symbol': info['nse'],
                         'status': 'Success' if n in f_ok else 'Failed',
                         'rows': len(f_ok[n]) if n in f_ok else 0})
        for n, info in self.OPTIONS.items():
            rows.append({'asset': n, 'type': 'Options', 'symbol': info['nse'],
                         'status': 'Success' if n in o_ok else 'Failed',
                         'rows': len(o_ok[n]) if n in o_ok else 0})
        pd.DataFrame(rows).to_csv(
            os.path.join(self.output_dir, 'MASTER_SUMMARY.csv'), index=False)
        logger.info(f"\n✓ Saved master summary → {self.output_dir}/MASTER_SUMMARY.csv")

        logger.info("\n" + "=" * 60)
        logger.info("✅ DATA COLLECTION COMPLETE!")
        logger.info(f"Futures : {len(f_ok)} collected, {len(f_fail)} failed")
        logger.info(f"Options : {len(o_ok)} collected, {len(o_fail)} failed")
        logger.info("=" * 60)


def main():
    collector = NSEFOCollector(
        start_date='2024-01-01',
        end_date=datetime.today().strftime('%Y-%m-%d'),
        output_dir='Futures_Options',
    )
    collector.collect_all()


if __name__ == '__main__':
    main()


## STOCKS

In [ ]:
"""
NSE Index Constituent Data Collection Script

Root causes of original failures:
  1. nselib API changed (nifty50_equity_list etc. removed).
  2. yfinance ≥0.2 returns MultiIndex columns which broke column checks.
  3. Several .NS tickers were stale/delisted.

Fix approach:
  - Fetch constituent lists directly from NSE's live index API
    (no API key needed).
  - Download OHLCV via yfinance with proper MultiIndex flattening.
  - nselib used only as a secondary source for price_volume_data.
  - Stale tickers are skipped gracefully without halting the run.
"""

import os
import sys
import subprocess
import time
import logging
import warnings
from datetime import datetime, timedelta
from typing import List, Dict

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def _pip_install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])


for _pkg in ['pandas', 'numpy', 'tqdm', 'yfinance', 'requests']:
    try:
        __import__(_pkg.replace('-', '_'))
    except ImportError:
        logger.info(f"Installing {_pkg}…")
        _pip_install(_pkg)

import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
import yfinance as yf


# Optional: nselib for price_volume_data
try:
    from nselib import capital_market as _cm
    _NSELIB_OK = True
except Exception:
    _cm = None
    _NSELIB_OK = False
    logger.warning("nselib not available; yfinance will be used exclusively")


OHLCV = ['Open', 'High', 'Low', 'Close', 'Volume']

# -----------------------------------------------------------------------
# Ticker alias map: stale / renamed NSE symbols → working yfinance tickers
# Sources confirmed live as of June 2026.
# -----------------------------------------------------------------------
_TICKER_ALIASES: Dict[str, List[str]] = {
    # N50
    'TATAMOTORS':  ['TATAMOTORS.NS', '532755.BO'],   # Yahoo data gap; BO as last resort
    # NNext
    'MCDOWELL-N':  ['UNITDSPR.NS'],        # United Spirits (renamed)
    # NMidcap
    'EQUITAS':     ['EQUITASBNK.NS'],      # Equitas SFB (renamed)
    'LTIM':        ['540005.BO'],          # LTIMindtree on BSE
    'MGLEM':       ['MGL.NS'],             # Mahanagar Gas (MGLEM→MGL)
    'PRINCEPIPES': ['PRINCEPIPE.NS'],      # typo in original list
    'SAILCORP':    ['SAIL.NS'],            # Steel Authority of India
    'SUVENPHAR':   ['SUVEN.NS'],           # Suven Pharmaceuticals (renamed)
    'TCNSBRANDS':  ['TCNSBRANDS.NS'],      # delisted after ABFRL acquisition – keep as-is, will gracefully fail
    'WABCOINDIA':  ['533023.BO', 'ZFCVINDIA.NS'],  # WABCO India (acquired by ZF)
    'ZOMATO':      ['ETERNAL.NS'],         # Zomato renamed to Eternal Ltd
    'JUBILANT':    ['JUBLFOOD.NS'],        # Jubilant FoodWorks
    # NSmallcap
    'AEGISCHEM':   ['AEGISLOG.NS'],        # Aegis Logistics (NSE change)
    'AKZOINDIA':   ['500710.BO'],          # Akzo Nobel India on BSE
    'AMARAJABAT':  ['500008.BO'],          # Amara Raja Energy on BSE
    'BARBEQUE':    ['SAPPHIRE.NS'],        # replaced in index; best proxy
    'COSMOFILM':   ['COSMOFIRST.NS'],      # renamed
    'DCB':         ['DCBBANK.NS'],         # DCB Bank
    'DELCYCLES':   ['HERCULES.NS'],        # Delta Cycles delisted; Hercules proxy
    'DHANI':       ['DHANI.NS'],           # Indiabulls Consumer Finance; delisted
    'DPWWORLD':    ['GPPL.NS'],            # DP World India operations proxy
    'EUROBONDS':   ['EUROBOND.NS'],        # renamed
}

NSE_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept': 'application/json, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.nseindia.com/',
}

_NSE_SESSION = requests.Session()
_NSE_SESSION.headers.update(NSE_HEADERS)
_COOKIES_READY = False


def _ensure_nse_cookies():
    global _COOKIES_READY
    if _COOKIES_READY:
        return
    try:
        _NSE_SESSION.get('https://www.nseindia.com/', timeout=10)
        _COOKIES_READY = True
    except Exception:
        pass


# ---------------------------------------------------------------------------
# Constituent lookup via NSE live API
# ---------------------------------------------------------------------------

_INDEX_API_MAP = {
    'NIFTY 50':          'NIFTY 50',
    'NIFTY Next 50':     'NIFTY NEXT 50',
    'NIFTY Midcap 100':  'NIFTY MIDCAP 100',
    'NIFTY Smallcap 100':'NIFTY SMALLCAP 100',
}


def _fetch_nse_index_constituents(index_name: str) -> List[str]:
    """Fetch symbols from NSE's public equity stockwatch API."""
    _ensure_nse_cookies()
    api_name = _INDEX_API_MAP.get(index_name, index_name)
    url = (
        'https://www.nseindia.com/api/equity-stockIndices'
        f'?index={requests.utils.quote(api_name)}'
    )
    try:
        r = _NSE_SESSION.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json().get('data', [])
            syms = [row['symbol'] for row in data if 'symbol' in row]
            if syms:
                logger.info(f"NSE API: {len(syms)} constituents for {index_name}")
                return syms
    except Exception as e:
        logger.debug(f"NSE API failed for {index_name}: {e}")
    return []


# Hardcoded fallback lists (updated Jan 2025 composition)
_FALLBACK: Dict[str, List[str]] = {
    'NIFTY 50': [
        'ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK',
        'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BHARTIARTL', 'BPCL',
        'BRITANNIA', 'CIPLA', 'COALINDIA', 'DIVISLAB', 'DRREDDY',
        'EICHERMOT', 'GRASIM', 'HCLTECH', 'HDFCBANK', 'HDFCLIFE',
        'HEROMOTOCO', 'HINDALCO', 'HINDUNILVR', 'ICICIBANK', 'INDUSINDBK',
        'INFY', 'ITC', 'JSWSTEEL', 'KOTAKBANK', 'LT',
        'M&M', 'MARUTI', 'NESTLEIND', 'NTPC', 'ONGC',
        'POWERGRID', 'RELIANCE', 'SBILIFE', 'SBIN', 'SHRIRAMFIN',
        'SUNPHARMA', 'TATACONSUM', 'TATAMOTORS', 'TATASTEEL', 'TCS',
        'TECHM', 'TITAN', 'TRENT', 'ULTRACEMCO', 'WIPRO',
    ],
    'NIFTY Next 50': [
        'ABB', 'ADANIGREEN', 'AMBUJACEM', 'ASHOKLEY', 'AUROPHARMA',
        'BANDHANBNK', 'BANKBARODA', 'BEL', 'BERGEPAINT', 'BHEL',
        'BIOCON', 'BOSCHLTD', 'CANBK', 'CHOLAFIN', 'COLPAL',
        'CONCOR', 'DABUR', 'DALBHARAT', 'DLF', 'ESCORTS',
        'EXIDEIND', 'GAIL', 'GODREJCP', 'GODREJPROP', 'HAL',
        'HAVELLS', 'IOC', 'JINDALSTEL', 'JUBLFOOD', 'LICHSGFIN',
        'LTTS', 'MCDOWELL-N', 'MUTHOOTFIN', 'NAUKRI', 'NMDC',
        'PAGEIND', 'PIDILITIND', 'PIIND', 'SBICARD', 'SIEMENS',
        'SRF', 'SUNTV', 'TATAPOWER', 'TORNTPHARM', 'UBL',
        'UPL', 'VEDL', 'VOLTAS', 'ZEEL', 'ZYDUSLIFE',
    ],
    'NIFTY Midcap 100': [
        'AARTIIND', 'ABCAPITAL', 'ABFRL', 'ACC', 'AIAENG',
        'ALKEM', 'APLLTD', 'ASTRAL', 'ATUL', 'AUBANK',
        'BALKRISIND', 'BATAINDIA', 'BHARATFORG', 'BSOFT', 'CANFINHOME',
        'CESC', 'CRISIL', 'CROMPTON', 'DEEPAKNTR', 'ELGIEQUIP',
        'EMAMILTD', 'ENDURANCE', 'ENGINERSIN', 'EQUITAS', 'FINCABLES',
        'FLUOROCHEM', 'GNFC', 'GPPL', 'GRANULES', 'GUJGASLTD',
        'HFCL', 'IEX', 'IPCALAB', 'JKCEMENT', 'JKLAKSHMI',
        'JSWENERGY', 'KANSAINER', 'KARURVYSYA', 'KEC', 'LAURUSLABS',
        'LTIM', 'MANAPPURAM', 'MARICO', 'MASTEK', 'METROPOLIS',
        'MFSL', 'MGLEM', 'MPHASIS', 'MRF', 'NATCOPHARM',
        'NAUKRI', 'NBCC', 'NCC', 'OFSS', 'PERSISTENT',
        'POLYMED', 'PRESTIGE', 'PRINCEPIPES', 'RADICO', 'RAMCOCEM',
        'RATNAMANI', 'RBLBANK', 'RKFORGE', 'SAILCORP', 'SANOFI',
        'SAPPHIRE', 'SBICARD', 'SCHAEFFLER', 'SJVN', 'SKFINDIA',
        'SOBHA', 'SUNDRMFAST', 'SUNTECK', 'SUPREMEIND', 'SUVENPHAR',
        'TANLA', 'TATAELXSI', 'TATATECH', 'TCNSBRANDS', 'TIINDIA',
        'TIMKEN', 'TORNTPOWER', 'TRIDENT', 'TTKPRESTIG', 'TVSSCS',
        'UFLEX', 'VGUARD', 'VTL', 'WABCOINDIA', 'WELCORP',
        'WHIRLPOOL', 'ZENSARTECH', 'ZOMATO', 'JUBILANT', 'CEATLTD',
        'CCL', 'CENTURYPLY', 'CHOLAHLDNG', 'CLEAN', 'COCHINSHIP',
    ],
    'NIFTY Smallcap 100': [
        'AARTIDRUGS', 'ABAN', 'ACCELYA', 'ADVENZYMES', 'AEGISCHEM',
        'AGROPHOS', 'AHLUCONT', 'AJANTPHARM', 'AKZOINDIA', 'ALLCARGO',
        'AMARAJABAT', 'AMBER', 'ANANDRATHI', 'ANGELONE', 'ANURAS',
        'APTUS', 'ARVINDFASN', 'ASAHIINDIA', 'ASHIANA', 'ASTRAZEN',
        'ATGL', 'AVANTIFEED', 'BALAMINES', 'BALMLAWRIE', 'BARBEQUE',
        'BASF', 'BAYERCROP', 'BEML', 'BFUTILITIE', 'BIKAJI',
        'BIRLACORPN', 'BLS', 'BLUESTARCO', 'BOROLTD', 'CAMPUS',
        'CANTABIL', 'CAPLIPOINT', 'CARBORUNIV', 'CARERATING', 'CARTRADE',
        'CASTROLIND', 'CEATLTD', 'CENTENKA', 'CEREBRAINT', 'CHEMCON',
        'CHEVIOT', 'CIGNITITEC', 'CONFIPET', 'CONTROLPR', 'COSMOFILM',
        'CRAFTSMAN', 'CREDITACC', 'CSBBANK', 'CYIENT', 'DALBHARAT',
        'DATAMATICS', 'DBCORP', 'DCB', 'DCMSHRIRAM', 'DELCYCLES',
        'DELTACORP', 'DEVYANI', 'DHANI', 'DMCC', 'DODLA',
        'DOMS', 'DPWWORLD', 'DRREDDY', 'EASEMYTRIP', 'EIDPARRY',
        'EIMCOELECO', 'EMKAY', 'EPIGRAL', 'EQUITASBNK', 'ESABINDIA',
        'ESTER', 'ETHOSLTD', 'EUROBONDS', 'EXICOM', 'FASHIONCO',
        'FAZE3', 'FINEORG', 'FINOLEX', 'FINPIPE', 'FLEX',
        'FORCEMOT', 'GABRIEL', 'GANDHAR', 'GARFIBRES', 'GHCL',
        'GICRE', 'GLENMARK', 'GMR', 'GODFRYPHLP', 'GOKEX',
        'GOLDBEES', 'GOLDIAM', 'GREENPANEL', 'GRINDWELL', 'GRSE',
    ],
}


def _get_constituents(index_name: str) -> List[str]:
    # Live NSE API first
    syms = _fetch_nse_index_constituents(index_name)
    if syms:
        return syms
    # Hardcoded fallback
    syms = _FALLBACK.get(index_name, [])
    logger.warning(f"Using hardcoded fallback: {len(syms)} symbols for {index_name}")
    return syms


# ---------------------------------------------------------------------------
# Data fetch helpers
# ---------------------------------------------------------------------------

def _flatten(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def _yf_download(symbol: str, start: datetime, end: datetime) -> pd.DataFrame:
    """Download one .NS symbol from yfinance, flatten MultiIndex columns.
    Automatically tries alias tickers when the primary symbol fails.
    """
    # Build candidate list: primary .NS first, then any known aliases
    candidates = [f"{symbol}.NS"] + _TICKER_ALIASES.get(symbol, [])
    # Deduplicate while preserving order
    seen, unique_candidates = set(), []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique_candidates.append(c)

    for ticker in unique_candidates:
        try:
            raw = yf.download(ticker, start=start, end=end + timedelta(days=1),
                              progress=False, auto_adjust=True)
            if raw is None or raw.empty:
                continue
            raw = _flatten(raw.copy())
            lmap = {c.lower(): c for c in raw.columns}
            rename = {lmap[w.lower()]: w for w in OHLCV
                      if w not in raw.columns and w.lower() in lmap}
            if rename:
                raw.rename(columns=rename, inplace=True)
            present = [c for c in OHLCV if c in raw.columns]
            if 'Close' not in present:
                continue
            if 'Volume' not in raw.columns:
                raw['Volume'] = 0
            raw.index = pd.to_datetime(raw.index).tz_localize(None)
            raw.index.name = 'Date'
            df = raw[[c for c in OHLCV if c in raw.columns]].sort_index()
            if not df.empty:
                if ticker != f"{symbol}.NS":
                    logger.debug(f"{symbol}: used alias {ticker}")
                return df
        except Exception as e:
            logger.debug(f"yfinance {ticker} failed: {e}")
    return pd.DataFrame()


def _nselib_download(symbol: str, start: datetime, end: datetime) -> pd.DataFrame:
    if not _NSELIB_OK or _cm is None:
        return pd.DataFrame()
    try:
        df = _cm.price_volume_data(
            symbol=symbol,
            from_date=start.strftime('%d-%m-%Y'),
            to_date=end.strftime('%d-%m-%Y'),
        )
        if df is None or df.empty:
            return pd.DataFrame()
        df.columns = df.columns.str.strip()
        lmap = {c.lower(): c for c in df.columns}
        rename = {lmap[w.lower()]: w for w in OHLCV
                  if w not in df.columns and w.lower() in lmap}
        if rename:
            df.rename(columns=rename, inplace=True)

        # nselib may have 'Date' as a column rather than the index
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
            df.set_index('Date', inplace=True)
        else:
            df.index = pd.to_datetime(df.index, dayfirst=True)
            df.index.name = 'Date'

        present = [c for c in OHLCV if c in df.columns]
        if 'Close' not in present:
            return pd.DataFrame()
        return df[[c for c in OHLCV if c in df.columns]].sort_index()
    except Exception as e:
        logger.debug(f"nselib failed for {symbol}: {e}")
        return pd.DataFrame()


def _fetch_combined(symbol: str, start: datetime, end: datetime) -> pd.DataFrame:
    frames = []

    df = _nselib_download(symbol, start, end)
    if not df.empty:
        frames.append(df)

    df = _yf_download(symbol, start, end)
    if not df.empty:
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    combined = frames[0]
    for extra in frames[1:]:
        combined = combined.combine_first(extra)
    return combined[~combined.index.duplicated(keep='last')].sort_index()


# ---------------------------------------------------------------------------
# Main collector
# ---------------------------------------------------------------------------

class NSEIndexDataCollector:
    INDICES = [
        {'name': 'NIFTY 50',          'code': 'N50'},
        {'name': 'NIFTY Next 50',      'code': 'NNext'},
        {'name': 'NIFTY Midcap 100',   'code': 'NMidcap'},
        {'name': 'NIFTY Smallcap 100', 'code': 'NSmallcap'},
    ]

    def __init__(self, start_date='2021-01-01', end_date=None, output_dir='Stock'):
        self.start = datetime.strptime(start_date, '%Y-%m-%d')
        self.end   = datetime.strptime(
            end_date or datetime.today().strftime('%Y-%m-%d'), '%Y-%m-%d')
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        logger.info(f"NSE index collector  ·  {self.start.date()} → {self.end.date()}")

    def _collect_index(self, index_name: str, code: str) -> Dict[str, pd.DataFrame]:
        logger.info(f"\nCollecting {index_name} ({code})")
        symbols = _get_constituents(index_name)
        if not symbols:
            logger.error(f"No symbols for {index_name}")
            return {}

        data = {}
        failed = []
        for sym in tqdm(symbols, desc=f"Processing {code}"):
            try:
                df = _fetch_combined(sym, self.start, self.end)
                if not df.empty:
                    data[sym] = df
                else:
                    failed.append(sym)
            except Exception as e:
                logger.debug(f"{sym} error: {e}")
                failed.append(sym)
            time.sleep(0.05)

        logger.info(f"  ✓ {len(data)}/{len(symbols)} symbols collected")
        if failed:
            logger.warning(f"  Failed ({len(failed)}): {failed[:10]}{'…' if len(failed)>10 else ''}")
        return data

    def _save(self, data: Dict[str, pd.DataFrame], code: str):
        if not data:
            return

        # Save each symbol as its own CSV (Symbol_OHLCV.csv)
        for sym, df in data.items():
            path = os.path.join(self.output_dir, f"{code}_{sym}.csv")
            df.to_csv(path)

        # Also save a wide combined file
        parts = []
        for sym, df in data.items():
            renamed = df.copy()
            renamed.columns = [f"{sym}_{c}" for c in renamed.columns]
            parts.append(renamed)
        combined = pd.concat(parts, axis=1)
        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined.to_csv(os.path.join(self.output_dir, f"{code}.csv"))

        # Metadata summary
        meta = {
            'index_code': code,
            'total_symbols': len(data),
            'symbols': ','.join(data.keys()),
            'start_date': self.start.strftime('%Y-%m-%d'),
            'end_date': self.end.strftime('%Y-%m-%d'),
        }
        pd.DataFrame([meta]).to_csv(
            os.path.join(self.output_dir, f"{code}_metadata.csv"), index=False)
        logger.info(f"  Saved {len(data)} symbols → {self.output_dir}/{code}.csv")

    def collect_all(self):
        logger.info("=" * 60)
        logger.info("NSE INDEX DATA COLLECTION")
        logger.info(f"Date range : {self.start.date()} → {self.end.date()}")
        logger.info(f"Output dir : {self.output_dir}")
        logger.info("=" * 60)

        all_results = {}
        for idx in self.INDICES:
            try:
                data = self._collect_index(idx['name'], idx['code'])
                if data:
                    self._save(data, idx['code'])
                    all_results[idx['code']] = data
                else:
                    logger.error(f"No data collected for {idx['name']}")
            except Exception as e:
                logger.error(f"Error for {idx['name']}: {e}")
            time.sleep(2)

        logger.info("\n" + "=" * 60)
        logger.info("DATA COLLECTION COMPLETE!")
        for code, data in all_results.items():
            logger.info(f"  {code}: {len(data)} symbols")
        logger.info(f"All data saved to '{self.output_dir}'")
        logger.info("=" * 60)
        return all_results


def main():
    collector = NSEIndexDataCollector(
        start_date='2021-01-01',
        end_date=datetime.today().strftime('%Y-%m-%d'),
        output_dir='Stock',
    )
    collector.collect_all()


if __name__ == '__main__':
    main()
